# 🏆 Чемпионский пайплайн: 4-Fold CNN (ResNet-18) + 349 признаков (пол + потолок) через Стэкинг
### Достижение минимального MAE (~6.03) за счёт синергии нейросетевого ансамбля и точной геометрии

Архитектура решения:
1. **Уровень 1 (Ансамбль CNN)**: 4 модели ResNet-18, обученные по GroupKFold (`folds.csv`) с multi-task loss.
   - Формируют честные Out-Of-Fold (OOF) предсказания на обучающей выборке.
   - На тесте дают усреднённый прогноз с Test-Time Augmentation (TTA).
2. **Геометрический модуль (349 признаков)**:
   - 235 базовых признаков (RGB 8×8, текстура 4×4, гистограммы, синий цвет);
   - 42 признака свободного пола (`FloorUNet`);
   - 42 признака видимого потолка (`CeilingUNet`, IoU=0.778);
   - 30 взаимных cross-признаков (зазор пол-потолок, видимость дальнего края, симметрия).
3. **Уровень 2 (Стэкинг / Мета-модель)**: Ridge-регрессия объединяет честные прогнозы CNN и 349 признаков геометрии.
4. **Уровень 3 (Калибровка)**: Пороговая коррекция экстремальных значений (≤ 3.5% → 0.0%, ≥ 94.5% → 100.0%).

In [ ]:
# 1. Проверка GPU и установка библиотек
import torch
import sys
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip install -q albumentations timm segmentation-models-pytorch pandas opencv-python-headless scikit-learn matplotlib seaborn


In [ ]:
# 2. Настройки обучения и файловая структура проекта
import shutil
import time
from pathlib import Path

# ==============================================================================
# ⚙️ НАСТРОЙКИ: ПЕРЕСЧИТАТЬ ВСЕ ВЕСА ЗАНОВО ИЛИ ИСПОЛЬЗОВАТЬ СТАРЫЕ?
# ==============================================================================
# RETRAIN_CNN = True  -> Обучить все 4 фолда CNN ResNet-18 заново с нуля на GPU (~6-8 мин)!
#                        Старые чекпоинты из архива удаляются/игнорируются.
# RETRAIN_CNN = False -> Использовать готовые чекпоинты из архива.
RETRAIN_CNN = True

# RECOMPUTE_GEOMETRY = True  -> Очистить кэш и извлечь 349 признаков сегментации заново (~2-3 мин).
# RECOMPUTE_GEOMETRY = False -> Использовать кэшированные признаки (если уже считались).
RECOMPUTE_GEOMETRY = False

# Гиперпараметры обучения 4 фолдов CNN (ResNet-18):
PHASE1_EPOCHS = 6    # Фаза 1: прогрев головы (backbone заморожен)
PHASE2_EPOCHS = 18   # Фаза 2: fine-tuning всей сети целиком
BATCH_SIZE = 32      # Размер батча (оптимально для GPU T4 / P100)
IMG_SIZE = 320       # Разрешение входного изображения 320x320
NUM_WORKERS = 4      # Потоков загрузки данных
# ==============================================================================

KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
WORK_SRC = WORK_DIR / "src"
MODELS_DIR = WORK_DIR / "models"
OUTPUT_DIR = WORK_DIR / "output_csv"
CACHE_DIR = WORK_DIR / "feature_cache"

for d in (MODELS_DIR, OUTPUT_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

if RECOMPUTE_GEOMETRY:
    print("🧹 Очистка кэша геометрических признаков...")
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Поиск корня датасета
dataset_candidates = []
for p in KAGGLE_INPUT.rglob("folds.csv"):
    if "DataSet" in p.parts:
        idx = p.parts.index("DataSet")
        root = Path(*p.parts[:idx])
        if root not in dataset_candidates:
            dataset_candidates.append(root)

if not dataset_candidates:
    for p in KAGGLE_INPUT.rglob("train_split.csv"):
        if "DataSet" in p.parts:
            idx = p.parts.index("DataSet")
            root = Path(*p.parts[:idx])
            if root not in dataset_candidates:
                dataset_candidates.append(root)

if not dataset_candidates:
    raise FileNotFoundError("Не найден датасет с DataSet в /kaggle/input!")

DATASET_ROOT = dataset_candidates[0]
print(f"DATASET_ROOT: {DATASET_ROOT}")

# Копирование исходного кода src
src_candidates = [p for p in KAGGLE_INPUT.rglob("src") if (p / "baseline.py").is_file() and "working" not in str(p)]
if src_candidates:
    SRC_SOURCE = src_candidates[0]
    print(f"Копируем код из {SRC_SOURCE} в {WORK_SRC}...")
    if WORK_SRC.exists(): shutil.rmtree(WORK_SRC)
    shutil.copytree(SRC_SOURCE, WORK_SRC)
else:
    if (DATASET_ROOT / "src").exists():
        shutil.copytree(DATASET_ROOT / "src", WORK_SRC)
    else:
        raise FileNotFoundError("Исходный код src не найден в /kaggle/input!")

# Управление чекпоинтами CNN
dest_cnn = MODELS_DIR / "cnn_v1"
if RETRAIN_CNN:
    if dest_cnn.exists(): shutil.rmtree(dest_cnn)
    dest_cnn.mkdir(parents=True, exist_ok=True)
    print("⚡ РЕЖИМ ПЕРЕОБУЧЕНИЯ (RETRAIN_CNN = True): чекпоинты будут обучены заново на GPU!")
else:
    if not dest_cnn.exists() or not list(dest_cnn.glob("fold*_best.pt")):
        for cnn_dir in KAGGLE_INPUT.rglob("cnn_v1"):
            if dest_cnn.exists(): shutil.rmtree(dest_cnn)
            shutil.copytree(cnn_dir, dest_cnn)
            print(f"Скопированы готовые чекпоинты CNN из {cnn_dir}")
            break

print(f"WORK_SRC: {WORK_SRC}")
import os
os.environ["PYTHONPATH"] = str(WORK_SRC)


In [ ]:
# 3. Проверка путей и весов моделей
FOLDS_CSV = DATASET_ROOT / "DataSet/train/folds.csv"
TRAIN_IMAGES = DATASET_ROOT / "DataSet/train/images"
TEST_CSV = DATASET_ROOT / "DataSet/test/test.csv"
TEST_IMAGES = DATASET_ROOT / "DataSet/test/images"

CNN_MODELS_DIR = MODELS_DIR / "cnn_v1"
CNN_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Веса сегментаторов (кузов, пол, потолок) - фиксированные геометрические экстракторы
TRUCK_WEIGHTS = WORK_SRC / "manual/truck_segmentation/models/best_unet_resnet18.pth"
FLOOR_WEIGHTS = WORK_SRC / "manual/floor_segmentation/models/floor_unet_resnet18_lr1e3.best_loss.pt"
CEILING_WEIGHTS = WORK_SRC / "manual/roof_segmentation/models/ceiling_unet_resnet18.best_iou.pt"

required_paths = {
    "folds_csv": FOLDS_CSV,
    "train_images": TRAIN_IMAGES,
    "test_csv": TEST_CSV,
    "test_images": TEST_IMAGES,
    "truck_weights": TRUCK_WEIGHTS,
    "floor_weights": FLOOR_WEIGHTS,
    "ceiling_weights": CEILING_WEIGHTS,
}

all_ok = True
for name, path in required_paths.items():
    status = "OK" if path.exists() else "НЕ НАЙДЕН"
    print(f"{name:16s}: {path} -> {status}")
    if not path.exists(): all_ok = False

if not all_ok:
    raise FileNotFoundError("Не все обязательные файлы найдены!")

ckpts = sorted(list(CNN_MODELS_DIR.glob("fold*_best.pt")))
print(f"\nРежим работы: {'[ОБУЧЕНИЕ С НУЛЯ]' if RETRAIN_CNN else '[ГОТОВЫЕ ВЕСА]'}")
print(f"Чекпоинтов CNN в папке: {len(ckpts)}/4")
for cp in ckpts:
    print(f"  - {cp.name} ({cp.stat().st_size / (1024*1024):.1f} MB)")


In [ ]:
# 4. Обучение 4-фолдового ансамбля CNN (ResNet-18) с нуля на GPU
import subprocess
import time

ckpts = sorted(list(CNN_MODELS_DIR.glob("fold*_best.pt")))
if len(ckpts) == 4 and not RETRAIN_CNN:
    print("Все 4 фолда CNN уже готовы! Пропускаем этап повторного обучения.")
else:
    print(f"🚀 Запуск обучения 4 фолдов CNN (ResNet-18) с нуля на {DEVICE.upper()}...")
    print(f"   Параметры: Phase 1 = {PHASE1_EPOCHS} эпох, Phase 2 = {PHASE2_EPOCHS} эпох, Batch = {BATCH_SIZE}, Size = {IMG_SIZE}")
    
    # Очищаем директорию чекпоинтов перед чистым обучением
    for cp in CNN_MODELS_DIR.glob("*.pt"):
        cp.unlink()
    if (CNN_MODELS_DIR / "metrics.json").exists():
        (CNN_MODELS_DIR / "metrics.json").unlink()

    train_cnn_cmd = [
        sys.executable, str(WORK_SRC / "CNN/train_cnn.py"),
        "--folds", str(FOLDS_CSV),
        "--images", str(TRAIN_IMAGES),
        "--output-dir", str(CNN_MODELS_DIR),
        "--backbone", "resnet18",
        "--img-size", str(IMG_SIZE),
        "--batch-size", str(BATCH_SIZE),
        "--num-workers", str(NUM_WORKERS),
        "--phase1-epochs", str(PHASE1_EPOCHS),
        "--phase2-epochs", str(PHASE2_EPOCHS),
    ]
    if DEVICE == "cuda":
        train_cnn_cmd.append("--amp")
    print("Команда запуска:", " ".join(train_cnn_cmd))
    
    start_t = time.time()
    env = dict(os.environ, PYTHONPATH=str(WORK_SRC))
    subprocess.run(train_cnn_cmd, env=env, check=True)
    elapsed_m = (time.time() - start_t) / 60
    print(f"\n✅ Обучение 4 фолдов CNN успешно завершено за {elapsed_m:.1f} мин!")

# Вывод метрик по фолдам CNN
metrics_file = CNN_MODELS_DIR / "metrics.json"
if metrics_file.exists():
    with open(metrics_file, "r", encoding="utf-8") as f:
        metrics = json.load(f)
    print("\n" + "=" * 50)
    print("📊 РЕЗУЛЬТАТЫ 4-FOLD CNN (ResNet-18):")
    print("=" * 50)
    for f_id, score in metrics.get("folds", {}).items():
        print(f"   Fold {f_id}: val MAE = {score:.4f} п.п.")
    print(f"   Mean Val MAE: {metrics.get('mean_val_mae', 0):.4f} ± {metrics.get('std_val_mae', 0):.4f} п.п.")
    print("=" * 50)


In [ ]:
# 5. Обучение СТЭКИНГА: OOF-предсказания CNN + 349 признаков геометрии -> Ridge
STACKING_MODEL_PATH = MODELS_DIR / "stacking_model.pkl"
STACKING_ERRORS_PATH = MODELS_DIR / "stacking_oof_errors.csv"

stacking_train_cmd = [
    sys.executable, str(WORK_SRC / "baseline.py"), "train",
    "--method", "stacking",
    "--folds", str(FOLDS_CSV),
    "--images", str(TRAIN_IMAGES),
    "--model", str(STACKING_MODEL_PATH),
    "--cnn-models-dir", str(CNN_MODELS_DIR),
    "--errors-output", str(STACKING_ERRORS_PATH),
    "--feature-cache", str(CACHE_DIR),
    "--truck-weights", str(TRUCK_WEIGHTS),
    "--floor-weights", str(FLOOR_WEIGHTS),
    "--ceiling-weights", str(CEILING_WEIGHTS),
    "--device", DEVICE,
    "--batch-size", "16",
]

print("Запуск обучения стэкинга:", " ".join(stacking_train_cmd))
subprocess.run(stacking_train_cmd, check=True)

# Вывод отчёта стэкинга
import json
import pandas as pd
report_path = STACKING_MODEL_PATH.with_suffix(".json")
if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as f:
        rep = json.load(f)
    print("\n" + "=" * 60)
    print("🏆 ИТОГОВЫЙ ОТЧЁТ СТЭКИНГА:")
    print("=" * 60)
    print(f"  Исходный CNN 4-fold OOF MAE: {rep.get('cnn_raw_oof_mae', 0):.4f} п.п.")
    print(f"  CNN с калибровкой OOF MAE:   {rep.get('cnn_calibrated_oof_mae', 0):.4f} п.п.")
    print(f"  Стэкинг (+349 фичей) MAE:    {rep.get('stacking_oof_mae', 0):.4f} п.п.")
    print(f"  Финальный MAE с калибровкой: {rep.get('final_calibrated_oof_mae', 0):.4f} п.п.")
    print(f"  Точность within 10%:         {rep.get('within_10_pct', 0):.2f}%")
    print("=" * 60)


In [ ]:
# 6. Инференс чемпионского стэкинга на тестовой выборке (test.csv)
SUBMISSION_PATH = OUTPUT_DIR / "submission_stacking.csv"

predict_cmd = [
    sys.executable, str(WORK_SRC / "baseline.py"), "predict",
    "--model", str(STACKING_MODEL_PATH),
    "--test-csv", str(TEST_CSV),
    "--images", str(TEST_IMAGES),
    "--output", str(SUBMISSION_PATH),
    "--feature-cache", str(CACHE_DIR),
    "--device", DEVICE,
    "--batch-size", "16",
    "--calibrate",
]

print("Запуск инференса:", " ".join(predict_cmd))
subprocess.run(predict_cmd, check=True)


In [ ]:
# 7. Проверка сформированного файла submission_stacking.csv
import numpy as np
if SUBMISSION_PATH.exists():
    sub_df = pd.read_csv(SUBMISSION_PATH)
    print(f"Строк в предсказании: {len(sub_df)} (ожидается 307)")
    print(f"Количество пропусков (NaN): {sub_df.isna().sum().to_dict()}")
    print(f"Минимум load_pct: {sub_df['load_pct'].min():.2f}%")
    print(f"Максимум load_pct: {sub_df['load_pct'].max():.2f}%")
    print("\nПервые 10 строк предсказаний:")
    display(sub_df.head(10))
    
    # Сверка с эталоном 6.03 (если есть в папке)
    for ref_name in ["submission_603.csv", "score_60373.csv", "submissionWW.csv"]:
        for ref_file in KAGGLE_INPUT.rglob(ref_name):
            ref_df = pd.read_csv(ref_file)
            corr = np.corrcoef(sub_df['load_pct'].values, ref_df['load_pct'].values)[0, 1]
            diff = np.abs(sub_df['load_pct'].values - ref_df['load_pct'].values).mean()
            print(f"\nСверка с {ref_name}:")
            print(f"  Корреляция:      {corr:.5f}")
            print(f"  Средняя разница: {diff:.4f} п.п.")
            break
else:
    print("Файл submission не найден!")
